In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. Read data
raw_path = "../data/raw_job_posts.csv"
processed_path = "../data/processed_job_posts.csv"

df = pd.read_csv(raw_path, encoding="utf-8-sig")

# 2. Quick view
display(df.head())
print("Shape:", df.shape)

# 3. Data info
df.info()

# 4. Missing values
missing = df.isna().sum()
display(missing[missing > 0])

# 5. Standardize column names
df.columns = df.columns.str.strip().str.lower()

# 6. Fill missing and strip text
text_cols = [
    "job_title", "company", "location", "source",
    "employment_type", "role_group", "salary",
    "description", "requirements", "skills_text", "posted_date", "url"
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

# 7. Fill default values for important columns
if "location" in df.columns:
    df["location"] = df["location"].replace("", "Unknown")

if "salary" in df.columns:
    df["salary"] = df["salary"].replace("", "Not specified")

if "role_group" in df.columns:
    df["role_group"] = df["role_group"].replace("", "Software / Other")

# 8. Drop duplicates
before = df.shape[0]

duplicate_subset = ["job_title", "company", "description"]
duplicate_subset = [col for col in duplicate_subset if col in df.columns]

df = df.drop_duplicates(subset=duplicate_subset)

after = df.shape[0]
print(f"Removed duplicates: {before - after}")
print(f"Remaining rows: {after}")

# 9. Create full_text
required_text_cols = ["job_title", "description", "requirements", "skills_text"]

for col in required_text_cols:
    if col not in df.columns:
        df[col] = ""

df["full_text"] = (
    df["job_title"] + " " +
    df["description"] + " " +
    df["requirements"] + " " +
    df["skills_text"]
).str.lower()

# 10. Create skill flags
skills = [
    "sql",
    "python",
    "excel",
    "power bi",
    "tableau",
    "pandas",
    "numpy",
    "scikit-learn",
    "machine learning",
    "deep learning",
    "etl",
    "postgresql",
    "mysql",
    "mongodb",
    "spark",
    "pytorch",
    "tensorflow",
    "hugging face",
    "dashboard",
    "data visualization"
]

for skill in skills:
    col_name = "skill_" + skill.replace(" ", "_").replace("-", "_")
    df[col_name] = df["full_text"].str.contains(skill, case=False, na=False)

# 11. Skill counts
skill_cols = [col for col in df.columns if col.startswith("skill_")]
skill_counts = df[skill_cols].sum().sort_values(ascending=False)

display(skill_counts)

# 12. Plot top 10 skills
top_skills = skill_counts.head(10)

plt.figure(figsize=(10, 5))
top_skills.sort_values().plot(kind="barh")
plt.title("Top 10 Required Skills in Internship Job Posts")
plt.xlabel("Number of Job Posts")
plt.ylabel("Skill")
plt.show()

# 13. Export processed data
df.to_csv(processed_path, index=False, encoding="utf-8-sig")

print("Saved processed data to:", processed_path)
print("Final shape:", df.shape)